# RAGNAR — offline benchmark analysis

What this notebook does with the runs currently in `results/`: turn three completed
offline index builds into the comparisons a per-stage RAG benchmark is supposed to
support, and flag the places where the numbers do **not** mean what they look like.

**Run status at time of writing**

| job | config | arm | status |
|---|---|---|---|
| 119585 | `configs/baseline.yaml` | fixed_word / MiniLM / FAISS-flat | complete |
| 119586 | `configs/chunk.yaml` | sentence / MiniLM / FAISS-flat | complete |
| 119587 | `configs/embed.yaml` | fixed_word / mpnet-dot / FAISS-flat | complete |
| 119624 | `configs/vector_db.yaml` | fixed_word / MiniLM / **Chroma** | hardware trace only — no index, no build trace |
| — | online phase | all arms | **not yet run** |

Everything below globs `results/`, so re-running the notebook after the Chroma build and
the online runs land picks them up with no edits. Sections 9 and 10 are already wired for
those files and degrade to a "not present yet" message.

**The three headline findings, up front**

1. The `sentence` arm is not comparable to the `fixed_word` arm at the vector level — its
   chunks are ~10x longer than the embedder's token window, so most of its corpus text is
   never embedded (section 3). Any retrieval-quality difference the online phase finds
   between these two arms is confounded with silent truncation.
2. The `embed` arm's config declares `metric: cosine` for a `*-dot-v1` model. The build
   trace proves the raw vectors are unnormalised (mean norm ~6.8 vs ~1.0), and FAISS then
   L2-normalises them — the magnitude signal that model is trained to use is discarded
   (section 6). A defensible choice, but it has to be stated as one.
3. Measured build time is 3-9 seconds; job wall-clock is ~19 minutes. Over 99% of each job
   is environment setup, so the hardware CSVs are almost entirely idle samples. At a 3 s
   interval the monitor catches 0-3 points inside the actual work — the sentence arm has
   **zero**. The hardware traces cannot support a utilisation or energy claim (section 7).

In [ ]:
from pathlib import Path
import json, re, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

ROOT    = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULTS = ROOT / "results"
PLOTS   = ROOT / "analysis" / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

# --- Assumptions made explicit, because several results below depend on them ---
# Rough chars-per-token for English Wikipedia prose under a WordPiece/BPE tokenizer.
# Used ONLY to estimate truncation exposure (section 3); every number derived from it is
# labelled "est.". Sensitivity to this constant is checked there.
CHARS_PER_TOKEN = 4.0

# Token window past which SentenceTransformer silently truncates.
# VERIFY THESE ON THE CLUSTER: SentenceTransformerEmbedder exposes .max_seq_length --
# print it in the next offline run and hard-code what it actually reports. The values
# below are HuggingFace card defaults and mpnet in particular varies by release.
MAX_SEQ_TOKENS  = {"all-MiniLM-L6-v2": 256, "multi-qa-mpnet-base-dot-v1": 512}
DEFAULT_MAX_TOK = 256
EMBED_DIM       = {"all-MiniLM-L6-v2": 384, "multi-qa-mpnet-base-dot-v1": 768}

print("results dir:", RESULTS)
print(*sorted(p.name for p in RESULTS.glob("*")), sep="\n  ")

## 1. Run inventory and provenance

In [ ]:
def load_runs(results=RESULTS):
    # One row per offline build: provenance + resolved config + build trace.
    # Keyed on job_id so an index config with no build trace (a crashed or still-running
    # job) still appears, with NaN metrics -- that is how 119624 shows up.
    rows = {}
    for cfgp in sorted(results.glob("*.index_config.json")):
        p = json.loads(cfgp.read_text(encoding="utf-8"))
        r = p["resolved"]
        job = str(p["job_id"])
        rows[job] = {
            "job_id":         job,
            "config":         Path(p["config_source"]).name,
            "backend":        r["index"]["database"],
            "index_type":     r["index"]["type"],
            "chunker":        r["chunker"]["type"],
            "chunk_size":     r["chunker"]["size"],
            "overlap":        r["chunker"]["overlap"],
            "embedder":       r["embedder"]["model"],
            "metric":         r["embedder"]["metric"],
            "prepend_titles": r["offline"]["prepend_titles"],
            "batch_size":     r["offline"]["embed_batch_size"],
            "data_path":      Path(r["offline"]["data_path"]).name,
            "git_sha":        p["git_sha"][:8],
            "git_dirty":      p["git_dirty"],
            "overrides":      ", ".join(p["overrides"]) or "-",
        }
    for tracep in sorted(results.glob("offline_*.json")):
        m = re.search(r"_(\d+|\d{8}_\d{6})\.json$", tracep.name)
        if not m:
            continue
        job = m.group(1)
        t = json.loads(tracep.read_text(encoding="utf-8"))
        rows.setdefault(job, {"job_id": job})
        rows[job].update({
            "trace_file":      tracep.name,
            "n_passages":      t["n_passages"],
            "n_chunks":        t["n_chunks"],
            "chunk_avg_chars": t["chunk_size_avg"],
            "chunk_min_chars": t["chunk_size_min"],
            "chunk_max_chars": t["chunk_size_max"],
            # read/chunk/unaccounted exist only in traces written after per-stage
            # chunk timing landed; .get keeps older traces loadable.
            "read_ms":         t["latency_ms"].get("read"),
            "chunk_ms":        t["latency_ms"].get("chunk"),
            "unaccounted_ms":  t["latency_ms"].get("unaccounted"),
            "embed_ms":        t["latency_ms"]["embed"],
            "index_ms":        t["latency_ms"]["index"],
            "total_ms":        t["latency_ms"]["total"],
            "chunks_per_s":    t["embed_throughput"]["chunks_per_sec"],
            "n_batches":       len(t["batches"]),
        })
    df = pd.DataFrame(rows.values()).sort_values("job_id").reset_index(drop=True)
    df["arm"] = df.apply(
        lambda r: f'{r["job_id"]} {r.get("chunker","?")}/'
                  f'{str(r.get("embedder","?")).split("-")[0]}/{r.get("backend","?")}',
        axis=1)
    return df


def load_batches(results=RESULTS):
    # Per-batch rows, flattened. One batch per `offline.file_chunk_size` documents.
    out = []
    for tracep in sorted(results.glob("offline_*.json")):
        m = re.search(r"_(\d+|\d{8}_\d{6})\.json$", tracep.name)
        if not m:
            continue
        t = json.loads(tracep.read_text(encoding="utf-8"))
        for b in t["batches"]:
            row = {"job_id": m.group(1), "batch_idx": b["batch_idx"],
                   "timestamp": pd.to_datetime(b["timestamp"]),
                   "is_training_batch": b["is_training_batch"],
                   "n_passages": b["n_passages"], "n_chunks": b["n_chunks"],
                   "n_skipped": b["n_skipped"], "skip_rate": b["skip_rate"],
                   "chunks_per_s": b["embed_throughput"]["chunks_per_sec"]}
            row.update({f"len_{k}": v for k, v in b["chunk_length"].items()})
            row.update({f"norm_{k}": v for k, v in b["embed_norm"].items()})
            out.append(row)
    return pd.DataFrame(out)


runs    = load_runs()
batches = load_batches()

runs[["job_id", "config", "chunker", "chunk_size", "embedder", "metric", "backend",
      "index_type", "n_passages", "n_chunks", "git_sha", "git_dirty"]]

Two provenance facts worth carrying into the writeup:

- **`git_dirty = True` on every build.** The recorded SHA does not reproduce these runs.
  The provenance machinery is doing its job by saying so; a thesis-grade sweep needs a
  clean tree, or the dirty diff archived alongside the index.
- **`n_batches == 1` everywhere.** `file_chunk_size: 1000` over a 1000-article corpus
  means the whole build is a single batch, so per-batch variance is unavailable and every
  throughput number below is a single sample with no error bar. Fine for a smoke test,
  not enough for a comparison claim. See section 11.

## 2. Corpus conservation and chunk geometry

First establish that the arms are chunking the *same* corpus. Total characters across all
chunks should be near-identical between arms that differ only in chunker; small
differences are legitimate (joining tokens, and the `min_chunk_words` merge rule).

In [ ]:
g = runs.dropna(subset=["n_chunks"]).copy()
g["total_chunk_chars"]  = g["n_chunks"] * g["chunk_avg_chars"]
g["chunks_per_article"] = g["n_chunks"] / g["n_passages"]
g["chars_per_article"]  = g["total_chunk_chars"] / g["n_passages"]

geo = g[["arm", "chunker", "chunk_size", "n_passages", "n_chunks", "chunks_per_article",
         "chunk_avg_chars", "chunk_min_chars", "chunk_max_chars",
         "total_chunk_chars", "chars_per_article"]].copy()
geo["total_chunk_chars"]  = geo["total_chunk_chars"].map(lambda v: f"{v:,.0f}")
geo["chars_per_article"]  = geo["chars_per_article"].round(0)
geo["chunks_per_article"] = geo["chunks_per_article"].round(2)
display(geo)

tot = g["n_chunks"] * g["chunk_avg_chars"]
spread = (tot.max() - tot.min()) / tot.mean()
print(f"\ncorpus char spread across arms: {spread:.3%} "
      f"({'consistent - same corpus' if spread < 0.02 else 'INVESTIGATE'})")

In [ ]:
# Chunk-length distribution from the five order statistics the trace records.
# Not a boxplot in the quartile sense: whiskers are min/max, the bar is p5-p95.
b = batches.merge(runs[["job_id", "arm", "chunker"]], on="job_id")

fig, ax = plt.subplots(figsize=(9, 3.2))
for i, (_, r) in enumerate(b.iterrows()):
    ax.hlines(i, r.len_min, r.len_max, color="0.75", lw=2, zorder=1)
    ax.barh(i, r.len_p95 - r.len_p5, left=r.len_p5, height=0.42,
            color="#4C72B0", alpha=0.85, zorder=2)
    ax.plot(r.len_mean, i, "o", color="#C44E52", ms=7, zorder=3)
    ax.text(r.len_max * 1.05, i, f"max {r.len_max:,.0f}", va="center", fontsize=8, color="0.35")
ax.set_yticks(range(len(b)))
ax.set_yticklabels(b["arm"])
ax.set_xscale("log")
ax.set_xlabel("chunk length (characters, log scale)")
ax.set_title("Chunk length: min-max (grey), p5-p95 (bar), mean (dot)")
ax.set_xlim(30, b["len_max"].max() * 3)
fig.tight_layout()
fig.savefig(PLOTS / "chunk_length_distribution.png", dpi=150)
plt.show()

**Read:** `fixed_word` does exactly what it promises — a tight p5-p95 band (~515-696
chars) around a 100-word target, with the min/max tail explained by the `min_chunk_words`
merge and by short articles. `sentence` at `size: 100` is a different object entirely:
p95 ~14,000 characters, max ~19,700.

The reason is config semantics, not a bug. `chunker.size` means **words** for `fixed_word`
and **sentences** for `sentence`, and one shared value of `100` was used for both. 100
sentences is roughly a whole Wikipedia article — hence 1.6 chunks per article for the
sentence arm versus 17.1 for fixed_word. As a benchmark grid this is a unit collision: the
two arms are not matched on any dimension a reader would call "chunk size".

## 3. Truncation exposure — the result that invalidates a naive chunker comparison

`SentenceChunker.warn_if_truncated` prints a warning for exactly this reason. Here is what
that warning is worth quantitatively: the embedder truncates at `max_seq_length` tokens,
so any chunk text past that point is stored in the database but **never enters its own
vector**. It is retrievable only through the prefix that fitted.

`est_tokens` is `chars / CHARS_PER_TOKEN`; *coverage* is the share of chunk characters
surviving into the vector (1.0 for a chunk inside the window). The mean-based figure is an
**upper bound** for the sentence arm: applying the cap to the mean overstates coverage
relative to the true mean of `min(len, cap)/len` over a right-skewed length distribution.

In [ ]:
def coverage_row(r):
    max_tok  = MAX_SEQ_TOKENS.get(r["embedder"], DEFAULT_MAX_TOK)
    cap_char = max_tok * CHARS_PER_TOKEN
    return pd.Series({
        "max_seq_tokens":   max_tok,
        "cap_chars":        cap_char,
        "est_tokens_mean":  round(r["chunk_avg_chars"] / CHARS_PER_TOKEN, 0),
        "est_tokens_p95":   round(r["len_p95"] / CHARS_PER_TOKEN, 0),
        "coverage_at_mean": round(min(cap_char, r["chunk_avg_chars"]) / r["chunk_avg_chars"], 3),
        "coverage_at_p95":  round(min(cap_char, r["len_p95"]) / r["len_p95"], 3),
    })


t = runs.dropna(subset=["n_chunks"]).merge(batches[["job_id", "len_p5", "len_p95"]], on="job_id")
trunc = pd.concat([t[["arm", "chunker", "embedder", "chunk_avg_chars"]],
                   t.apply(coverage_row, axis=1)], axis=1)
display(trunc)

t["cap_chars"] = t["embedder"].map(
    lambda m: MAX_SEQ_TOKENS.get(m, DEFAULT_MAX_TOK) * CHARS_PER_TOKEN)
t["embedded_chars_ub"] = t["n_chunks"] * np.minimum(t["cap_chars"], t["chunk_avg_chars"])
t["total_chars"]       = t["n_chunks"] * t["chunk_avg_chars"]

print("\nCorpus-level embedded-text coverage (upper bound):")
for _, r in t.iterrows():
    print(f"  {r['arm']:<34} {r['embedded_chars_ub'] / r['total_chars']:6.1%}"
          f"   ({r['embedded_chars_ub']/1e6:5.2f}M of {r['total_chars']/1e6:5.2f}M chars)")

In [ ]:
# Sensitivity: does the conclusion survive different chars-per-token and token windows?
rows = []
for cpt in (3.5, 4.0, 4.5, 5.0):
    for maxtok in (128, 256, 384, 512):
        for _, r in t.iterrows():
            cap = maxtok * cpt
            rows.append({"chars_per_token": cpt, "max_seq_tokens": maxtok, "arm": r["arm"],
                         "coverage": min(cap, r["chunk_avg_chars"]) / r["chunk_avg_chars"]})

sens = (pd.DataFrame(rows)
        .pivot_table(index="arm", columns=["max_seq_tokens", "chars_per_token"], values="coverage"))
try:
    # Heatmap needs jinja2; plain table is the fallback so the notebook has no extra dep.
    display(sens.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1, axis=None))
except (AttributeError, ImportError):
    display(sens.round(2))

print("\nworst-case coverage per arm across the whole grid:")
print(sens.min(axis=1).round(3).to_string())

**Read:** at the real 256-token window the `fixed_word` arms are at full coverage for every
chars-per-token value in the grid — 100 words is comfortably inside it. (They only dip, to
~0.74, in the hypothetical 128-token column, which no model here uses.) The `sentence` arm
sits at 14-20% at 256 tokens and never exceeds ~39% anywhere in the grid, including the
most generous corner (512 tokens, 5 chars/token). The conclusion does not depend on the
constant.

**What this means for the benchmark.** If the online phase reports lower recall for the
sentence arm, that is *not* evidence that sentence-boundary chunking is worse than
fixed-size chunking. It is a measurement of an embedder truncating 100-sentence chunks. To
make the arms comparable, `chunker.size` for `sentence` has to land the resulting chunks
inside the token window — roughly 3-6 sentences puts the mean near the ~600-char
fixed_word target. Two defensible designs; pick one and state it:

- **Match on chunk length** (sentence size ~4): isolates "does respecting sentence
  boundaries help, holding chunk size roughly constant".
- **Keep sentence size large and switch to a long-context embedder**: turns the arm into a
  chunk-size study, not a boundary study.

The current setting answers neither question.

## 4. Throughput: chunks/s is the wrong unit for cross-arm comparison

The trace reports `chunks_per_sec`. Across arms with wildly different chunk lengths that
is not a work rate — the sentence arm looks 3x "slower" per chunk while doing the same
total character volume. Normalising by characters and by articles gives the comparison a
stable denominator.

In [ ]:
th = runs.dropna(subset=["n_chunks"]).copy()
th["embed_s"]        = th["embed_ms"] / 1000
th["chars_per_s"]    = (th["n_chunks"] * th["chunk_avg_chars"]) / th["embed_s"]
th["articles_per_s"] = th["n_passages"] / th["embed_s"]
th["ms_per_chunk"]   = th["embed_ms"] / th["n_chunks"]
th["ms_per_article"] = th["embed_ms"] / th["n_passages"]
# Builds made before per-stage chunk timing landed have no read/chunk split, so
# everything outside embed and index is one residual. Newer traces measure read and
# chunk directly and the residual shrinks to trace bookkeeping only.
th["has_chunk_timer"] = th.get("chunk_ms", pd.Series(index=th.index, dtype=float)).notna()
th["read_ms_f"]  = th.get("read_ms",  pd.Series(index=th.index, dtype=float)).fillna(0.0)
th["chunk_ms_f"] = th.get("chunk_ms", pd.Series(index=th.index, dtype=float)).fillna(0.0)
th["other_ms"]   = (th["total_ms"] - th["embed_ms"] - th["index_ms"]
                    - th["read_ms_f"] - th["chunk_ms_f"])
print("per-stage chunk timing present:",
      ", ".join(f"{r.arm}={'yes' if r.has_chunk_timer else 'no'}" for _, r in th.iterrows()))

display(th[["arm", "embedder", "chunker", "n_chunks", "read_ms_f", "chunk_ms_f",
            "embed_ms", "index_ms", "other_ms", "total_ms",
            "chunks_per_s", "chars_per_s", "articles_per_s", "ms_per_article"]]
        .round({"chunks_per_s": 0, "chars_per_s": 0, "articles_per_s": 1, "ms_per_article": 2}))

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
titles = ["chunks / s\n(what the trace reports)",
          "characters / s\n(work-normalised)",
          "articles / s\n(corpus-normalised)"]
for ax, col, title in zip(axes, ["chunks_per_s", "chars_per_s", "articles_per_s"], titles):
    ax.bar(th["arm"], th[col], color=["#4C72B0", "#DD8452", "#55A868"][:len(th)])
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
fig.suptitle("Embedding throughput under three denominators", fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS / "throughput_denominators.png", dpi=150)
plt.show()

In [ ]:
# Where the measured build time actually goes.
fig, ax = plt.subplots(figsize=(8.5, 3))
left = np.zeros(len(th))
for col, color, label in [("read_ms_f",  "#937860", "read + parse"),
                          ("chunk_ms_f", "#DD8452", "chunk"),
                          ("embed_ms",   "#4C72B0", "embed"),
                          ("index_ms",   "#C44E52", "index add"),
                          ("other_ms",   "#8C8C8C", "unattributed")]:
    ax.barh(th["arm"], th[col], left=left, color=color, label=label)
    left = left + th[col].values
ax.set_xlabel("ms")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Offline build time decomposition (measured section only)")
fig.tight_layout()
fig.savefig(PLOTS / "build_time_decomposition.png", dpi=150)
plt.show()

print(th[["arm", "read_ms_f", "chunk_ms_f", "embed_ms", "index_ms", "other_ms", "total_ms"]]
      .assign(embed_pct=lambda d: (d.embed_ms / d.total_ms * 100).round(1),
              index_pct=lambda d: (d.index_ms / d.total_ms * 100).round(2),
              other_pct=lambda d: (d.other_ms / d.total_ms * 100).round(1))
      .to_string(index=False))

**Read:**

- **Embedding dominates the measured build** (~78-94% of `total_ms`); FAISS `flat`
  insertion is 0.03-0.6%. For a flat index that is expected — `add` is a memcpy. It will
  *not* hold for `ivf_pq` at 36M vectors, where training and PQ encoding become real
  costs. "Indexing is free" is a finding about `index.type: flat`, not about FAISS.
- **The mpnet arm is ~17% slower per chunk than MiniLM** on identical chunks (119585 vs
  119587) — far less than the ~2x you would predict from 768 vs 384 dimensions and a much
  larger model. On an H100 at this batch size the GPU is not the bottleneck and the run is
  too short for a steady-state rate to emerge. Treat this ratio as unusable until the
  job-overhead problem in section 7 is fixed.
- **The sentence arm has the largest non-embed cost** despite producing 11x fewer chunks.
  For the three builds above that is inferred from a residual, because they predate
  per-stage chunk timing. Builds made after it landed report `latency_ms.chunk` directly,
  and the bar chart splits read from chunk automatically — at which point "sentence
  splitting costs more per article than word windowing" becomes a measured per-stage
  result rather than a subtraction. A synthetic check of the new timers put sentence
  chunking at roughly an eighth the throughput of fixed-word chunking.

## 5. Index footprint — bytes on disk versus bytes of vector

The saved index is two files: the FAISS `.faiss` (vectors) and the `.pkl` (chunk texts and
ids). The second is the surprise — it is not a "vector database size" in the sense a
reader would assume.

In [ ]:
def index_files(job, results=RESULTS):
    out = {}
    for p in results.glob(f"*_{job}.index*"):
        if p.name.endswith("_config.json"):
            continue
        out[p.name] = p.stat().st_size
    return out


fp = []
for _, r in runs.dropna(subset=["n_chunks"]).iterrows():
    files   = index_files(r["job_id"])
    faiss_b = sum(v for k, v in files.items() if k.endswith(".faiss"))
    pkl_b   = sum(v for k, v in files.items() if k.endswith(".pkl"))
    dim     = EMBED_DIM.get(r["embedder"], np.nan)
    fp.append({"arm": r["arm"], "n_chunks": int(r["n_chunks"]), "dim": dim,
               "faiss_MB": faiss_b / 1e6, "pkl_MB": pkl_b / 1e6,
               "total_MB": (faiss_b + pkl_b) / 1e6,
               "faiss_B_per_vec": faiss_b / r["n_chunks"] if faiss_b else np.nan,
               "theoretical_B_per_vec": dim * 4,
               "pkl_B_per_chunk": pkl_b / r["n_chunks"] if pkl_b else np.nan,
               "chunk_avg_chars": r["chunk_avg_chars"]})

fp = pd.DataFrame(fp)
fp["faiss_overhead_B"]   = fp["faiss_B_per_vec"] - fp["theoretical_B_per_vec"]
fp["pkl_bytes_per_char"] = fp["pkl_B_per_chunk"] / fp["chunk_avg_chars"]
display(fp.round(2))

print("\nSanity: a flat fp32 index should be dim*4 bytes/vector plus a small header.")
for _, r in fp.iterrows():
    print(f"  {r['arm']:<34} {r['faiss_B_per_vec']:8.1f} B/vec  "
          f"(theoretical {r['theoretical_B_per_vec']:.0f}, "
          f"overhead {r['faiss_overhead_B']:+.1f} B)")
print("\nPayload pickle vs vector index:")
for _, r in fp.iterrows():
    print(f"  {r['arm']:<34} vectors {r['faiss_MB']:6.1f} MB   "
          f"payload {r['pkl_MB']:6.1f} MB   payload/vectors = {r['pkl_MB']/r['faiss_MB']:.2f}x")

**Read:**

- The `.faiss` files land within a byte or two of `dim x 4` per vector, confirming `flat`
  stores raw fp32 and that the fp16 model output is cast back to float32 before FAISS sees
  it, exactly as `SentenceTransformerEmbedder.embed` documents.
- **The payload `.pkl` is ~10-11 MB in every arm regardless of chunk count** — it holds the
  same corpus text either way. In the sentence arm it is over 4x the size of the vector
  index. When reporting "index size" for the vector-DB comparison (section 9), separate
  vector bytes from payload bytes or the Chroma-vs-FAISS result mostly measures how each
  backend serialises text.
- Per stored character the pickle costs ~1 byte, so payload scales with corpus text while
  the vector index scales with chunk *count* x dimension. Those scale differently, which is
  the whole reason chunk size is an interesting knob.

## 6. Embedding-norm diagnostic: the cosine/dot mismatch

`WikipediaLoader` records the norm distribution of the **raw** model output, before FAISS's
`normalize_L2`. That makes it a direct probe of whether the configured metric matches how
the model was trained.

In [ ]:
nm = batches.merge(runs[["job_id", "arm", "embedder", "metric"]], on="job_id")
nm["normalised_output"] = (nm["norm_mean"].sub(1.0).abs() < 0.01) & (nm["norm_std"] < 0.01)
nm["metric_matches_model"] = ~(nm["embedder"].str.contains("dot", case=False)
                               & nm["metric"].eq("cosine"))
display(nm[["arm", "embedder", "metric", "norm_mean", "norm_std", "norm_min", "norm_max",
            "norm_p5", "norm_p95", "normalised_output", "metric_matches_model"]].round(4))

fig, ax = plt.subplots(figsize=(8.5, 2.8))
for i, (_, r) in enumerate(nm.iterrows()):
    ax.hlines(i, r.norm_min, r.norm_max, color="0.75", lw=2)
    ax.barh(i, r.norm_p95 - r.norm_p5, left=r.norm_p5, height=0.4, color="#4C72B0", alpha=0.85)
    ax.plot(r.norm_mean, i, "o", color="#C44E52", ms=7)
ax.axvline(1.0, ls="--", color="#55A868", lw=1.2, label="unit norm (cosine-ready)")
ax.set_yticks(range(len(nm)))
ax.set_yticklabels(nm["arm"])
ax.set_xlabel("raw embedding L2 norm (before FAISS normalize_L2)")
ax.set_title("Raw output norms by arm")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS / "embedding_norms.png", dpi=150)
plt.show()

for _, r in nm.iterrows():
    if not r["metric_matches_model"]:
        print(f"FLAG  {r['arm']}: model '{r['embedder']}' emits norms with mean "
              f"{r['norm_mean']:.3f} (std {r['norm_std']:.3f}), i.e. relevance is partly "
              f"encoded in magnitude, but metric='{r['metric']}' L2-normalises it away.")

**Read:** MiniLM emits unit vectors (mean 1.0000, std 0.0002) — `metric: cosine` is a no-op
for it and is correct. `multi-qa-mpnet-base-dot-v1` emits norms with mean ~6.82, std ~0.15,
range 5.54-7.18: a real magnitude signal, which is the point of a `-dot-v1` model. The
config's own comment says as much, and the run still used `cosine`.

Two consequences for the embed arm:

1. **The comparison as run is "MiniLM-cosine vs mpnet-cosine".** That is a legitimate
   experiment — it isolates the encoder, holding the metric fixed — but it is not "MiniLM
   vs mpnet as each is intended to be used". Say which one you are claiming.
2. **For the intended comparison**, add a `metric: dot` build of mpnet. One extra offline
   run turns the arm into a 2x2 (model x metric), which is a stronger result than either
   single comparison: it separates "is mpnet a better encoder" from "does the magnitude
   signal carry retrieval value".

The norm spread also yields a free prediction to test online: under `dot`, longer chunks
(larger norms) should be systematically favoured. That length bias is worth measuring, not
just noting.

## 7. Hardware telemetry — and why these CSVs are mostly idle

The monitor samples every ~3 s for the life of the *job*. The build lasts a few seconds.
Aligning samples to the build window (from the batch timestamp in the trace, plus
`total_ms`) shows how little of each trace is actually the workload.

In [ ]:
def load_hw(results=RESULTS):
    frames = []
    for p in sorted(results.glob("hardware_offline_*.csv")):
        job = re.search(r"_(\d+|\d{8}_\d{6})\.csv$", p.name).group(1)
        d = pd.read_csv(p, parse_dates=["timestamp"])
        d["job_id"] = job
        frames.append(d)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


hw = load_hw()

# Build window: the batch starts at its recorded timestamp and the measured build lasts
# total_ms. With one batch per run this window is the entire workload.
win = (batches.groupby("job_id")["timestamp"].min().rename("build_start").to_frame()
       .join(runs.set_index("job_id")["total_ms"]))
win["build_end"] = win["build_start"] + pd.to_timedelta(win["total_ms"], unit="ms")

span = hw.groupby("job_id")["timestamp"].agg(["min", "max", "count"])
span["job_wall_s"]        = (span["max"] - span["min"]).dt.total_seconds()
span["sample_interval_s"] = span["job_wall_s"] / (span["count"] - 1)
span = span.join(win)
span["build_s"]         = span["total_ms"] / 1000
span["build_pct_of_job"] = span["build_s"] / span["job_wall_s"] * 100
span["samples_in_build"] = [
    int(((hw.job_id == j) & (hw.timestamp >= r["build_start"]) & (hw.timestamp <= r["build_end"])).sum())
    if pd.notna(r["build_start"]) else 0
    for j, r in span.iterrows()]

_cols = ["min", "max", "count", "job_wall_s", "sample_interval_s", "build_s",
         "build_pct_of_job", "samples_in_build"]
_num  = ["job_wall_s", "sample_interval_s", "build_s", "build_pct_of_job"]
display(span[_cols].astype({c: float for c in _num}).round({c: 3 for c in _num}))

In [ ]:
jobs = sorted(hw["job_id"].unique())
fig, axes = plt.subplots(len(jobs), 1, figsize=(11, 2.2 * len(jobs)))
axes = np.atleast_1d(axes)
for ax, job in zip(axes, jobs):
    d  = hw[hw.job_id == job]
    t0 = d["timestamp"].min()
    x  = (d["timestamp"] - t0).dt.total_seconds()
    ax.plot(x, d["dcgmi_power_w"], color="#C44E52", lw=1, label="GPU power (W)")
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(x, d["gpu_mem_used_mb"] / 1000, color="#4C72B0", lw=1, label="GPU mem (GB)")
    ax2.set_ylabel("GB", fontsize=8)
    if job in win.index and pd.notna(win.loc[job, "build_start"]):
        ax.axvspan((win.loc[job, "build_start"] - t0).total_seconds(),
                   (win.loc[job, "build_end"]   - t0).total_seconds(),
                   color="#55A868", alpha=0.45, label="measured build")
    ax.set_title(str(runs.set_index("job_id")["arm"].get(job, job)), fontsize=9, loc="left")
    ax.set_ylabel("W", fontsize=8)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left", ncol=3)
axes[-1].set_xlabel("seconds since first sample")
fig.suptitle("GPU power and memory over each job; green = the part that is the pipeline",
             fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS / "hardware_timeline.png", dpi=150)
plt.show()

In [ ]:
# Idle baseline vs in-build, for runs where the window is known.
METRICS = ["cpu_util_pct", "gpu_mem_used_mb", "dcgmi_power_w", "dcgmi_sm_active"]
rows = []
for job, r in span.iterrows():
    d = hw[hw.job_id == job]
    if pd.isna(r["build_start"]):
        rows.append({"job_id": job, "phase": "whole job (no build window)", "n": len(d),
                     **{c: d[c].mean() for c in METRICS}})
        continue
    inb = d[(d.timestamp >= r["build_start"]) & (d.timestamp <= r["build_end"])]
    pre = d[d.timestamp < r["build_start"]]
    for name, sub in [("setup/idle", pre), ("build", inb)]:
        if len(sub):
            rows.append({"job_id": job, "phase": name, "n": len(sub),
                         **{c: sub[c].mean() for c in METRICS}})

phase = pd.DataFrame(rows).merge(runs[["job_id", "arm"]], on="job_id", how="left")
display(phase[["arm", "job_id", "phase", "n"] + METRICS].round(2))

print("\nPeak GPU memory over the whole job (a high-water mark, so robust to sampling rate):")
print(hw.groupby("job_id")["gpu_mem_used_mb"].max().rename("peak_MB").round(0).to_string())

**Read — this is a measurement-validity problem, not a result:**

- **Build time is well under 1% of job wall-clock.** ~19 minutes of job for ~3-9 seconds of
  pipeline. The rest is module import, model download/load and CUDA context setup. Any
  energy or utilisation number computed over the whole CSV is a number about SLURM startup,
  not about RAG.
- **0-3 samples land inside the build window.** Job 119586 has **none**: its 2.7 s build
  finished between two 3 s samples, so its "build" phase row below is empty and every
  in-build statistic for that arm is undefined, not merely noisy. Where samples do land
  (2 for 119585, 3 for 119587) you still cannot characterise the workload: no ramp, the
  peak is missed, and mean SM activity is decided by whichever sample happened to fall
  inside. Nothing in this section supports a utilisation or energy comparison.
- **Peak GPU memory does separate the arms meaningfully** — ~4.1 GB for MiniLM vs ~12.6 GB
  for mpnet, ~3x, tracking model size and 768-dim activations. This is the one hardware
  figure here I would quote, and it is robust to the sampling problem precisely because it
  is a high-water mark rather than an instantaneous rate.
- Jobs 119586 and 119587 ran **concurrently** (overlapping wall-clock windows). Their GPU
  memory series differ, so they were on separate devices and are not contaminating each
  other — but confirm node assignment before publishing any cross-arm hardware comparison.

**Fixes.** (b) is done: `monitor.sh` now samples at a true 1 s. It previously asked for 1 s
and delivered 3.0 s because `mpstat` and `iostat` each block for a full interval before
returning; CPU, RAM and disk now come from `/proc` deltas and `nvidia-smi` runs as a
background stream, so no per-tick call blocks. **Every CSV above predates that change and
is still 3.0 s** — the tables in this section describe the old cadence and should not be
re-read as if they were 1 s data.

Still outstanding, in order of value: (a) run the offline phase over enough corpus that the
build takes minutes rather than seconds — the single change that makes every hardware and
throughput number meaningful, and which no sampling rate can substitute for; (c) emit
explicit build-start/build-end markers into the hardware CSV instead of reconstructing the
window from batch timestamps.

## 8. Extrapolating to the full corpus

With a single 1000-article batch these projections are indicative only — one sample, no
variance, and a short-run rate that has not reached steady state (section 7). They are
still worth computing now, because they say whether the planned full run is hours or days
before you queue it.

In [ ]:
TARGET_ARTICLES = 6_000_000   # set to the article count of the full wiki dump
M_PQ            = 48          # index.m_pq from the configs, 8 bits per sub-quantiser

ex = runs.dropna(subset=["n_chunks"]).copy()
ex["scale"]           = TARGET_ARTICLES / ex["n_passages"]
ex["proj_chunks"]     = ex["n_chunks"] * ex["scale"]
ex["proj_embed_h"]    = (ex["embed_ms"] / 1000) * ex["scale"] / 3600
ex["proj_total_h"]    = (ex["total_ms"] / 1000) * ex["scale"] / 3600
ex["dim"]             = ex["embedder"].map(EMBED_DIM)
ex["proj_flat_GB"]    = ex["proj_chunks"] * ex["dim"] * 4 / 1e9
# ivf_pq at the configured geometry: m_pq bytes/vector + ~16 B id and list overhead
ex["proj_ivfpq_GB"]   = ex["proj_chunks"] * (M_PQ + 16) / 1e9
ex["proj_payload_GB"] = ex["proj_chunks"] * ex["chunk_avg_chars"] * 1.0 / 1e9

display(ex[["arm", "proj_chunks", "proj_embed_h", "proj_total_h",
            "proj_flat_GB", "proj_ivfpq_GB", "proj_payload_GB"]]
        .assign(proj_chunks=lambda d: d.proj_chunks.map("{:,.0f}".format))
        .round(2))

GPU_GB = hw["gpu_mem_total_mb"].max() / 1000 if len(hw) else float("nan")
print(f"\nAt {TARGET_ARTICLES:,} articles (GPU on these nodes: {GPU_GB:.0f} GB):")
for _, r in ex.iterrows():
    print(f"  {r['arm']:<34} embed {r['proj_embed_h']:6.1f} h | "
          f"flat {r['proj_flat_GB']:7.1f} GB | "
          f"ivf_pq(m={M_PQ}) {r['proj_ivfpq_GB']:6.1f} GB | "
          f"payload {r['proj_payload_GB']:6.1f} GB")

**Read:** the flat-index projection is the operational punchline — at full corpus an fp32
flat index is an order of magnitude past the GPU memory these jobs see, while the
configured `ivf_pq` geometry brings it to something that fits. That is the justification
for the IVF-PQ choice expressed as a number rather than an assertion, and it is worth a
line in the thesis.

Read the sentence arm's projected embed hours with section 3 in hand: it is cheap partly
because it only embeds ~16% of the corpus text. A truncation-corrected sentence arm (fewer
sentences per chunk, hence more chunks) will project far closer to the fixed_word arm, so
do not bank the 3.7x embedding saving until that re-run exists.

It also shows why the chunker choice is a *systems* decision and not only a quality one:
the fixed_word arm projects ~11x the vectors of the sentence arm, hence ~11x the index, for
the same text. The right frame for the chunker comparison is quality-per-GB and
quality-per-embed-hour, not quality alone — a framing that only becomes computable once the
online phase lands.

Adjust `TARGET_ARTICLES` to the real dump size. Note the unit: the trace's `n_passages` is
*articles* from the reconstructed article corpus, not the ~36M passage records of the
original dump, so do not scale by 36M here without converting first.

## 9. Pending: the Chroma build (119624) and the vector-DB comparison

Job 119624 wrote a hardware trace and nothing else — no `offline_ChromaDB_119624.json`, no
index directory. The build did not complete. The cell below reports what is present, so
this section fills in automatically once the run succeeds.

When it lands, report FAISS vs Chroma on these axes, and **separate vector bytes from
payload bytes** (section 5) or the result mostly measures text serialisation:

| axis | source |
|---|---|
| build throughput | `latency_ms.embed` vs `latency_ms.index` — the index share is where the backends actually differ |
| on-disk footprint | vector store vs payload, separately |
| peak RSS / GPU mem | hardware CSV high-water mark |
| query latency | online phase, `stages.retrieval.avg_latency_ms` |
| recall parity | both are exact/flat here, so recall should be **identical**; any gap is a bug, not a finding |

In [ ]:
completed = set(runs.dropna(subset=["n_chunks"])["job_id"])
missing   = [j for j in hw["job_id"].unique() if j not in completed]
print("jobs with hardware telemetry but no completed build trace:", missing or "none")

chroma_traces = sorted(RESULTS.glob("offline_ChromaDB_*.json"))
chroma_stores = sorted(RESULTS.glob("ChromaDB_*")) + sorted((ROOT / "chroma_db").glob("*"))
if chroma_traces:
    print("\nChroma build trace(s) found - already included in every table above:")
    for p in chroma_traces:
        print("  ", p.name)
else:
    print("\nNo Chroma build trace yet. Once results/offline_ChromaDB_<job>.json exists,")
    print("every table above picks it up with no edits (load_runs globs 'offline_*.json').")
print("chroma store artifacts on disk:", [p.name for p in chroma_stores] or "none")

if missing:
    d = hw[hw.job_id.isin(missing)]
    print("\nWhat the incomplete job's telemetry shows "
          "(did it get as far as loading a model?):")
    print(d.groupby("job_id")[["gpu_mem_used_mb", "dcgmi_power_w", "cpu_util_pct"]]
            .agg(["max", "mean"]).round(1).to_string())
    print("\nFor reference, peak GPU memory of the completed MiniLM arms:")
    mini = runs[runs["embedder"].eq("all-MiniLM-L6-v2")]["job_id"]
    print(hw[hw.job_id.isin(mini)].groupby("job_id")["gpu_mem_used_mb"].max().round(0).to_string())

Peak GPU memory plus GPU power are the useful clues. If memory reaches the ~4 GB the MiniLM
arms reach *and* power rises above idle, the embedder loaded and ran, so the failure is
downstream in the Chroma backend. If power stays flat at idle, the job died before any
work reached the GPU and the problem is environment, not pipeline code.

**As it stands, 119624 is the second case:** peak GPU memory 2,544 MB against 4,054-4,172 MB
for the MiniLM arms, and DCGM power pinned at 63.4-63.5 W for the entire 20-minute job —
the same idle floor the other jobs show before their models load. No forward pass ever ran.
Look for the failure in the job's stderr around Chroma import or store creation, not in the
embedding path. The 2.5 GB of resident memory with no power draw is most likely CUDA
context plus another tenant on the device, not this job's model.

## 10. Pending: the online phase

Nothing in `results/` matches an online report yet. The loader below is written against the
schema `evaluate.py` emits, so it works the moment the first run lands.

Once online results exist, these are the comparisons the offline numbers above set up —
each one a claim the offline data alone cannot make:

1. **Quality per index-GB and per embed-hour.** Join `stages.retrieval.quality.recall_at_k`
   to section 8's `proj_flat_GB` / `proj_embed_h`. The sentence arm buys an ~11x smaller
   index; the question is what fraction of recall that costs.
2. **Truncation as a covariate, not a confound.** Section 3 predicts the sentence arm loses
   recall for reasons unrelated to boundary quality. If a re-run at a token-window-sized
   sentence chunk recovers the gap, that confirms section 3 and is a finding in its own
   right.
3. **Reranker contribution per arm.** `stages.reranking.{mrr,ndcg,recall}_delta` is already
   computed before/after. A cross-encoder scores the *stored text*, not the vector, so the
   sentence arm may show an unusually large rerank delta — the reranker recovering what the
   embedder truncated away. That is a genuinely interesting per-stage result this pipeline
   can measure directly.
4. **Latency budget end to end.** `stages.*.avg_latency_ms` across embedding / retrieval /
   reranking / generation, per arm. Generation will dominate; the interesting number is
   retrieval+rerank as a share, because that is the part the offline choices move.
5. **Pool health before anything else.** `retrieval.pool.n_short_pools` and
   `avg_top_k_returned`: with a 1000-article corpus and `top_k: 100`, short pools are
   likely, and `recall_at_k` is not interpretable while they are common. Check this first.
6. **The dot-vs-cosine 2x2** from section 6, if the extra mpnet build gets made.

In [ ]:
def load_online(results=RESULTS):
    # Load any online report emitted by evaluate.py. Returns (summary_df, raw_dict).
    reports, raw = [], {}
    for p in sorted(results.glob("*.json")):
        if p.name.startswith("offline_") or p.name.endswith("_config.json") or "hardware" in p.name:
            continue
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict) or "stages" not in d:
            continue
        raw[p.name] = d
        s   = d["stages"]
        ret = s.get("retrieval", {}) or {}
        q   = ret.get("quality", {}) or {}
        pool = ret.get("pool", {}) or {}
        rr  = s.get("reranking", {}) or {}
        gen = s.get("generation", {}) or {}
        reports.append({
            "file": p.name, "dataset": d.get("dataset"), "n": d.get("n_samples"),
            "total_s": d.get("total_elapsed_s"),
            "emb_ms": (s.get("embedding", {}) or {}).get("avg_latency_ms"),
            "ret_ms": ret.get("avg_latency_ms"),
            "rr_ms":  rr.get("avg_latency_ms"),
            "gen_ms": gen.get("avg_latency_ms"),
            "recall@k": q.get("recall_at_k"), "ndcg@k": q.get("ndcg_at_k"), "mrr": q.get("mrr"),
            "pool_recall@k": q.get("pool_recall_at_k"),
            "avg_pool": pool.get("avg_top_k_returned"), "n_short_pools": pool.get("n_short_pools"),
            "rr_ndcg_delta": rr.get("ndcg_delta"), "rr_mrr_delta": rr.get("mrr_delta"),
            "em": gen.get("avg_exact_match"), "rouge_l": gen.get("avg_rouge_l"),
            "tok_per_s": gen.get("avg_tokens_per_sec"),
            "ctx_overflow": gen.get("context_overflow_rate"),
            "failure_rate": (d.get("failures") or {}).get("failure_rate"),
        })
    return pd.DataFrame(reports), raw


online, online_raw = load_online()
if len(online):
    display(online)
    lat = online.set_index("file")[["emb_ms", "ret_ms", "rr_ms", "gen_ms"]].fillna(0)
    ax = lat.plot(kind="barh", stacked=True, figsize=(9, 0.7 * len(lat) + 1.6),
                  color=["#4C72B0", "#55A868", "#DD8452", "#C44E52"])
    ax.set_xlabel("avg latency per query (ms)")
    ax.set_title("Online stage latency budget")
    plt.tight_layout(); plt.savefig(PLOTS / "online_latency_budget.png", dpi=150); plt.show()
else:
    print("No online reports in results/ yet.")
    print("After:  sbatch run_rag_online.job results/FAISSDB_<job>.index configs/<arm>.yaml")
    print("this cell tabulates every arm side by side and plots the latency budget.")

## 11. What these numbers can and cannot support

**Defensible from the data as it stands**

- Chunker choice changes vector count by ~11x on identical text, and index size with it.
- Flat-index insertion is negligible next to embedding (for `flat` — says nothing about
  IVF-PQ, where training and PQ encoding are real costs).
- Peak GPU memory: MiniLM ~4.1 GB, mpnet ~12.6 GB, roughly 3x.
- A flat fp32 index of the full corpus does not fit in the GPU memory available; the
  configured IVF-PQ geometry does.
- MiniLM emits unit-norm vectors; `multi-qa-mpnet-base-dot-v1` does not, and the runs
  normalised that signal away.

**Not defensible yet, and why**

| claim | blocker | fix |
|---|---|---|
| "chunker X retrieves better than Y" | online phase not run; plus the truncation confound in section 3 | run online; re-run the sentence arm at a token-window-sized `size` |
| any throughput comparison | 3-9 s builds, one batch, no repeats, no steady state | larger corpus, 3+ repeats, report median and spread |
| any energy or utilisation figure | 2-4 hardware samples inside the build window | longer build **and** sub-second sampling |
| "mpnet is a better/worse encoder" | run as cosine, discarding the magnitude signal it is trained for | add a `metric: dot` build, giving a model x metric 2x2 |
| "FAISS vs Chroma" | 119624 never produced an index | re-run; separate vector bytes from payload bytes |
| reproducibility of any of it | `git_dirty: True` on all three builds | clean tree, or archive the diff next to the index |

**Cheapest changes with the largest effect on result quality**

1. Scale the offline corpus up until builds run for minutes. This one change fixes the
   throughput, hardware and steady-state problems at once.
2. Repeat each arm at least 3 times and report median with spread. Every number here is n=1.
3. Give `sentence` (and `paragraph`) a `size` in their own units that lands inside the
   embedder's token window, so the chunker arms compare boundary policy rather than
   truncation.
4. Commit the tree before the sweep so `git_sha` means something.